In [1]:
import os
import json
import pickle
import random
import numpy as np

In [2]:
import openai
openai.organization = os.environ['openai_organization'] 
openai.api_key = os.environ['openai_api_key']
#openai.Model.list()

# Load mapping

In [3]:
mapping = json.load(open("oct8/kgindex.json"))
entities = dict([(value, key) for key, value in mapping["e"].items()])
relations = \
    {0: 'is related to',
     1: 'has context of',
     2: 'is a',
     3: 'is synonym of',
     4: 'at location',
     5: 'is etymologically related to',
     6: 'is distinct from',
     7: 'has the last subevent',
     8: 'is used for',
     9: 'is similar to',
     10: 'desires',
     11: 'is antonym of',
     12: 'dbpedia',
     13: 'is a part of',
     14: 'is a form of',
     15: 'has a',
     16: 'is capable of',
     17: 'is an instance of',
     18: 'has a prerequisite of',
     19: 'is motivated by the goal of',
     20: 'is derived from',
     21: 'has subevent',
     22: 'causes',
     23: 'receives sanction by',
     24: 'has the property of',
     25: 'entails',
     26: 'has the first subevent',
     27: 'does not desire',
     28: 'causes desire',
     29: 'is made of',
     30: 'does not have the property of',
     31: 'is created by',
     32: 'is located near',
     33: 'is not capable of',
     34: 'is defined as',
     35: 'is a manner of'}

# Load Pickle files

In [4]:
path_profix = "oct8"

In [5]:
pickle_files = {}
for t in ["type0000",
          "type0001",
          "type0002",
          "type0004",
          "type0005",
          "type0008",
          "type0009",
          "type0007"]:
    path_to_pickle = path_profix + "/" + t + ".pickle"
    if os.path.isfile(path_to_pickle):
        if t not in pickle_files:
            pickle_files[t] = [path_to_pickle]
        else:
            pickle_files[t].append(path_to_pickle)
pickle_files

{'type0000': ['oct8/type0000.pickle'],
 'type0001': ['oct8/type0001.pickle'],
 'type0002': ['oct8/type0002.pickle'],
 'type0004': ['oct8/type0004.pickle'],
 'type0005': ['oct8/type0005.pickle'],
 'type0008': ['oct8/type0008.pickle'],
 'type0009': ['oct8/type0009.pickle']}

# Chatgpt Prompt

In [6]:
background_str ="""
[Goal]
Use your inherent knowledge to find the proper assignment of variables that mostly satisfies the given conditions.

[Background]
1. A variable is a placeholder that can be any concept in the world.
2. An assignment of a variable is to assign the concept to the variable.
3. Each condition involves two terms (concept or variable) and the relation between them.
4. Each condition has a [necessity value] that describes the necessary confidence. If the confidence value is greater than the necessity value, the condition is satisfied.
5. Each condition has an [importance value] that indicates the importance of satisfying this condition. 
6. Each assignment of variables will provide one or multiple evaluations for all conditions, by grounding the variables to the corresponding concepts. 
7. Each evaluation has a [confidence value] in the range of 0 to 1, where 0 means such evaluation is totally false, 1 means such evaluation is totally true. You can have your own judgment of [confidence value] based on your inherent knowledge about how confident such evaluation should be.
8. An evaluation is satisfied if its confidence value is greater than the necessity value that we provide.
9. For each satisfied evaluation, the assignment that causes this evaluation to gain a score, which is the result of the confidence value multiplied by the importance value.
10. In the end, you should compare the total scores of these provided candidate assignments to find the most suitable and proper candidate and output it to me. The most suitable and proper candidate is the candidate with the highest or maximal total score compared to other provided candidates in terms of assignments.


[Task input]
The input of this task contains three parts of declarations.
1. The statement of variables
2. The statement of conditions
3. The statement of the assignments to the variables

"""

format_sample_str = """
[1. The statement of variables]
We have the following variables
1. f1

[2. The statement of conditions]
1. f1 {relation1} {element1}; the necessity value is {alpha1}; the importance value is {beta1}. 
2. f1 {relation2} {element2}; the necessity value is {alpha2}; the importance value is {beta2}.

[3. The statement of assignments]
The possible value of f1 could be 
1 {choice1}
2 {choice2}
3 {choice3}
4 {choice4}

[Expected output]
Your output should be only the candidate index. DO NOT ADD YOUR EXPLANATION.
For example, if the answer is "2 test", your output MUST be the candidate index only. It MUST look like:
"2
<new line>
<new line>"

"""

# Parser

# type0000

In [7]:
dataset_type0000 = pickle.load(open(pickle_files["type0000"][0], "rb"))
len(dataset_type0000)

254

In [8]:
def parser_single(formula):
    last = 0
    
    relation = ""
    element1 = ""
    element2 = ""
    alpha = 0
    beta = 0
    
    for i in range(len(formula)):
        if formula[i] == "(":
            relation = formula[0:i]
            last = i+1
            break
    
    for i in range(last, len(formula)):   
        if formula[i] == ",":
            element1 = formula[last:i]
            last = i+1
            break
            
    for i in range(last, len(formula)):   
        if formula[i] == ",":
            element2 = formula[last:i]
            last = i+1
            break
    
    for i in range(last, len(formula)):   
        if formula[i] == ",":
            alpha = float(formula[last:i-1]) / 100
            last = i+1
            break
    
    for i in range(last, len(formula)):   
        if formula[i] == ")":
            beta = float(formula[last:i])
            last = i+1
            break
    return relation, element1, element2, alpha, beta

format_type0000_str = """
[1. The statement of variables]
We have the following variables
1. f1

[2. The statement of conditions]
1.  {s1} {r1} f1; the necessity value is {alpha1}; the importance value is {beta1}. 

[3. The statement of assignments]
The possible value of f1 could be 
1 {choice1}
2 {choice2}
3 {choice3}
4 {choice4}

[Expected output]
Your output should be only the candidate index. DO NOT ADD YOUR EXPLANATION.
For example, if the answer is "2 test", your output MUST be the candidate index only. It MUST look like:
"2
<new line>
<new line>"

"""

outputs = []
for row in dataset_type0000[:200]:
    # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
    idx = row[0]
    _, _, _, alpha1, beta1 = parser_single(row[1])
    r1 = entities[row[2]["r1"]]
    s1 = entities[row[2]["s1"]]
    
    choice1, choice2, choice3, choice4 = entities[row[3][0]],entities[row[3][1]], entities[row[3][2]], entities[row[3][3]]
    
    expected_answer = entities[row[5]]
    
    # Send Request to ChatGPT
    input_prompt = background_str + format_type0000_str.format(r1=r1,s1=s1,alpha1=alpha1,beta1=beta1,
                                                               choice1=choice1, choice2=choice2, choice3=choice3, choice4=choice4)
    
#     print(input_prompt)
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
                    {"role": "user", "content": input_prompt}
                ]
    )


    reply = response['choices'][0]['message']['content']
    
    answer = None
    if  "1" in reply:
        answer = row[3][0]
    elif "2" in reply:
        answer = row[3][1]
    elif "3" in reply:
        answer = row[3][2]
    elif "4" in reply:
        answer = row[3][3]
    
    print("id:{},".format(idx), "reply:{},".format(reply), "mapped answer:", answer)
    outputs.append((idx, answer))

pickle_file = open("oct8/type0000_outputs.pickle", "wb")
pickle.dump(outputs, pickle_file)
pickle_file.close()

id:0, reply:2, mapped answer: 4174
id:1, reply:2, mapped answer: 4115
id:2, reply:1, mapped answer: 912
id:3, reply:1, mapped answer: 236
id:4, reply:3, mapped answer: 12139
id:5, reply:4, mapped answer: 531
id:6, reply:1, mapped answer: 3068
id:7, reply:3, mapped answer: 1247
id:8, reply:1, mapped answer: 2121
id:9, reply:3, mapped answer: 690
id:10, reply:2, mapped answer: 1129
id:11, reply:2, mapped answer: 3002
id:12, reply:3, mapped answer: 1582
id:13, reply:1, mapped answer: 396
id:14, reply:2, mapped answer: 13821
id:15, reply:1, mapped answer: 1522
id:16, reply:3, mapped answer: 811
id:17, reply:1, mapped answer: 952
id:18, reply:4, mapped answer: 5070
id:19, reply:4, mapped answer: 7619
id:20, reply:1, mapped answer: 722
id:21, reply:4, mapped answer: 2121
id:22, reply:2, mapped answer: 1884
id:23, reply:1, mapped answer: 4031
id:24, reply:3, mapped answer: 13775
id:25, reply:2, mapped answer: 1530
id:26, reply:3, mapped answer: 2997
id:27, reply:2, mapped answer: 10311
id:28,

In [10]:
pickle_file = open("oct8/type0000_outputs.pickle", "rb")
test_db = pickle.load(pickle_file)
pickle_file.close()
test_db

[(0, 4174),
 (1, 4115),
 (2, 912),
 (3, 236),
 (4, 12139),
 (5, 531),
 (6, 3068),
 (7, 1247),
 (8, 2121),
 (9, 690),
 (10, 1129),
 (11, 3002),
 (12, 1582),
 (13, 396),
 (14, 13821),
 (15, 1522),
 (16, 811),
 (17, 952),
 (18, 5070),
 (19, 7619),
 (20, 722),
 (21, 2121),
 (22, 1884),
 (23, 4031),
 (24, 13775),
 (25, 1530),
 (26, 2997),
 (27, 10311),
 (28, 4136),
 (29, 3893),
 (30, 433),
 (31, 4181),
 (32, 11794),
 (33, 2349),
 (34, 33),
 (35, 913),
 (36, 2130),
 (37, 1063),
 (38, 2820),
 (39, 360),
 (40, 11791),
 (41, 788),
 (42, 3895),
 (43, 558),
 (44, 6455),
 (45, 965),
 (46, 3370),
 (47, 6274),
 (48, 2583),
 (49, 436),
 (50, 372),
 (51, 132),
 (52, 2937),
 (53, 3498),
 (54, 2872),
 (55, 8845),
 (56, 3111),
 (57, 6182),
 (58, 8167),
 (59, 1796),
 (60, 232),
 (61, 1318),
 (62, 331),
 (63, 4722),
 (64, 2788),
 (65, 146),
 (66, 1112),
 (67, 2565),
 (68, 8903),
 (69, 12244),
 (70, 3573),
 (71, 11358),
 (72, 173),
 (73, 1594),
 (74, 12862),
 (75, 8678),
 (76, 5568),
 (77, 558),
 (78, 7700)

In [9]:
outputs

[(0, 4174),
 (1, 4115),
 (2, 912),
 (3, 236),
 (4, 12139),
 (5, 531),
 (6, 3068),
 (7, 1247),
 (8, 2121),
 (9, 690),
 (10, 1129),
 (11, 3002),
 (12, 1582),
 (13, 396),
 (14, 13821),
 (15, 1522),
 (16, 811),
 (17, 952),
 (18, 5070),
 (19, 7619),
 (20, 722),
 (21, 2121),
 (22, 1884),
 (23, 4031),
 (24, 13775),
 (25, 1530),
 (26, 2997),
 (27, 10311),
 (28, 4136),
 (29, 3893),
 (30, 433),
 (31, 4181),
 (32, 11794),
 (33, 2349),
 (34, 33),
 (35, 913),
 (36, 2130),
 (37, 1063),
 (38, 2820),
 (39, 360),
 (40, 11791),
 (41, 788),
 (42, 3895),
 (43, 558),
 (44, 6455),
 (45, 965),
 (46, 3370),
 (47, 6274),
 (48, 2583),
 (49, 436),
 (50, 372),
 (51, 132),
 (52, 2937),
 (53, 3498),
 (54, 2872),
 (55, 8845),
 (56, 3111),
 (57, 6182),
 (58, 8167),
 (59, 1796),
 (60, 232),
 (61, 1318),
 (62, 331),
 (63, 4722),
 (64, 2788),
 (65, 146),
 (66, 1112),
 (67, 2565),
 (68, 8903),
 (69, 12244),
 (70, 3573),
 (71, 11358),
 (72, 173),
 (73, 1594),
 (74, 12862),
 (75, 8678),
 (76, 5568),
 (77, 558),
 (78, 7700)

# type0001

In [11]:
dataset_type0001 = pickle.load(open(pickle_files["type0001"][0], "rb"))
print(len(dataset_type0001))
dataset_type0001[:5]

390


[(0,
  '(r1(s1,e1,25%,0.6))&(r2(e1,f1,75%,0.6))',
  {'r1': 3, 'r2': 0, 's1': 13927},
  array([ 163, 2142, 2932, 1333]),
  array([0.9612, 0.9818, 1.0013, 1.0032]),
  1333),
 (1,
  '(r1(s1,e1,25%,0.6))&(r2(e1,f1,25%,0.6))',
  {'r1': 8, 'r2': 3, 's1': 2876},
  array([  13, 5681, 1978, 3264]),
  array([0.8512, 0.9612, 0.8512, 0.8512]),
  5681),
 (2,
  '(r1(s1,e1,75%,1.0))&(r2(e1,f1,25%,0.4))',
  {'r1': 0, 'r2': 0, 's1': 6763},
  array([12595,  9170,  5526,  1013]),
  array([0.9197, 0.993 , 1.0664, 1.1093]),
  1013),
 (3,
  '(r1(s1,e1,25%,0.6))&(r2(e1,f1,25%,0.5))',
  {'r1': 0, 'r2': 0, 's1': 1089},
  array([  259,  8792, 10383,  5898]),
  array([0.9165, 0.9241, 0.9256, 0.9413]),
  5898),
 (4,
  '(r1(s1,e1,25%,0.7))&(r2(e1,f1,75%,0.7))',
  {'r1': 0, 'r2': 2, 's1': 276},
  array([ 3858, 10483,  1328,  2914]),
  array([1.1214, 1.1822, 1.2036, 1.2573]),
  2914)]

In [12]:
def parser_type0001(formula):
    first_formula = formula[1:formula.index(")")+1]
    second_formula = formula[formula.index("&")+2:-1]
    
    r1, s1, e1, alpha1, beta1 = parser_single(first_formula)
    r2, e1, f1, alpha2, beta2 = parser_single(second_formula)
    
    return alpha1, beta1, alpha2, beta2


format_type0001_str = """
[1. The statement of variables]
We have the following variables
1. f1
2. e1

[2. The statement of conditions]
1.  {s1} {r1} e1; the necessity value is {alpha1}; the importance value is {beta1}. 
2.  e1 {r2} f1; the necessity value is {alpha2}; the importance value is {beta2}. 


[3. The statement of assignments]
The possible value of f1 could be 
1 {choice1}
2 {choice2}
3 {choice3}
4 {choice4}

[Expected output]
Your output should be only the candidate index. DO NOT ADD YOUR EXPLANATION.
For example, if the answer is "2 test", your output MUST be the candidate index only. It MUST look like:
"2
<new line>
<new line>"

"""

outputs = []
for row in dataset_type0001[:200]:
    # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
    idx = row[0]
    alpha1, beta1, alpha2, beta2 = parser_type0001(row[1])
    r1 = entities[row[2]["r1"]]
    r2 = entities[row[2]["r2"]]
    s1 = entities[row[2]["s1"]]
    
    choice1, choice2, choice3, choice4 = entities[row[3][0]],entities[row[3][1]], entities[row[3][2]], entities[row[3][3]]
    
    expected_answer = entities[row[5]]
    
    # Send Request to ChatGPT
    input_prompt = background_str + format_type0001_str.format(r1=r1,s1=s1,alpha1=alpha1,beta1=beta1,
                                                               r2=r2,alpha2=alpha2,beta2=beta2,
                                                               choice1=choice1, choice2=choice2, choice3=choice3, choice4=choice4)
    
#     print(input_prompt)
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
                    {"role": "user", "content": input_prompt}
                ]
    )


    reply = response['choices'][0]['message']['content']
    
    answer = None
    if  "1" in reply:
        answer = row[3][0]
    elif "2" in reply:
        answer = row[3][1]
    elif "3" in reply:
        answer = row[3][2]
    elif "4" in reply:
        answer = row[3][3]
    
    print("id:{}".format(idx), "reply: {}".format(reply), "mapped answer:", answer)
    outputs.append((idx, answer))
    
pickle_file = open("oct8/type0001_outputs.pickle", "wb")
pickle.dump(outputs, pickle_file)
pickle_file.close()

id:0 reply: 3 mapped answer: 2932
id:1 reply: 2 mapped answer: 5681
id:2 reply: 2 mapped answer: 9170
id:3 reply: 3 mapped answer: 10383
id:4 reply: 4 mapped answer: 2914
id:5 reply: 2 mapped answer: 741
id:6 reply: 2 mapped answer: 10383
id:7 reply: 3 mapped answer: 1314
id:8 reply: 2 mapped answer: 595
id:9 reply: 1 mapped answer: 862
id:10 reply: 1 mapped answer: 4853
id:11 reply: 1 mapped answer: 558
id:12 reply: 4 mapped answer: 2483
id:13 reply: 1 mapped answer: 356
id:14 reply: 4 mapped answer: 182
id:15 reply: 1 mapped answer: 3363
id:16 reply: 1 mapped answer: 70
id:17 reply: 4 mapped answer: 6411
id:18 reply: 4 mapped answer: 3370
id:19 reply: 3 mapped answer: 8845
id:20 reply: 2 mapped answer: 5702
id:21 reply: 1 mapped answer: 4266
id:22 reply: 4 mapped answer: 7551
id:23 reply: 2 mapped answer: 12598
id:24 reply: 2 mapped answer: 682
id:25 reply: 2 mapped answer: 13005
id:26 reply: 1 mapped answer: 832
id:27 reply: 3 mapped answer: 196
id:28 reply: 4 mapped answer: 7878
id

In [13]:
outputs

[(0, 2932),
 (1, 5681),
 (2, 9170),
 (3, 10383),
 (4, 2914),
 (5, 741),
 (6, 10383),
 (7, 1314),
 (8, 595),
 (9, 862),
 (10, 4853),
 (11, 558),
 (12, 2483),
 (13, 356),
 (14, 182),
 (15, 3363),
 (16, 70),
 (17, 6411),
 (18, 3370),
 (19, 8845),
 (20, 5702),
 (21, 4266),
 (22, 7551),
 (23, 12598),
 (24, 682),
 (25, 13005),
 (26, 832),
 (27, 196),
 (28, 7878),
 (29, 9741),
 (30, 80),
 (31, 2914),
 (32, 3443),
 (33, 8784),
 (34, 573),
 (35, 11),
 (36, 5877),
 (37, 6007),
 (38, 4302),
 (39, 6007),
 (40, 2497),
 (41, 3886),
 (42, 145),
 (43, 6699),
 (44, 1522),
 (45, 4802),
 (46, 659),
 (47, 887),
 (48, 507),
 (49, 3246),
 (50, 12064),
 (51, 7830),
 (52, 737),
 (53, 2925),
 (54, 5325),
 (55, 1838),
 (56, 5012),
 (57, 1016),
 (58, 10025),
 (59, 1517),
 (60, 7312),
 (61, 33),
 (62, 13098),
 (63, 1696),
 (64, 11555),
 (65, 12897),
 (66, 10285),
 (67, 644),
 (68, 284),
 (69, 1062),
 (70, 2846),
 (71, 12660),
 (72, 3581),
 (73, 2684),
 (74, 2121),
 (75, 7551),
 (76, 2937),
 (77, 2336),
 (78, 386)

# type0002

In [14]:
dataset_type0002 = pickle.load(open(pickle_files["type0002"][0], "rb"))
print(len(dataset_type0002))
dataset_type0002[:5]

23


[(0,
  '(r1(s1,f1,75%,0.4))&(r2(s2,f1,75%,0.8))',
  {'r1': 0, 'r2': 0, 's1': 1872, 's2': 5878},
  array([1952,   33, 2362, 8118]),
  array([0.8512, 0.9523, 0.8512, 0.8512]),
  33),
 (1,
  '(r1(s1,f1,25%,0.5))&(r2(s2,f1,25%,0.5))',
  {'r1': 8, 'r2': 8, 's1': 7699, 's2': 7699},
  array([ 497, 2351,  516, 1287]),
  array([0.7093, 1.    , 0.7093, 0.7093]),
  2351),
 (2,
  '(r1(s1,f1,25%,0.8))&(r2(s2,f1,75%,0.1))',
  {'r1': 0, 'r2': 0, 's1': 1583, 's2': 1583},
  array([ 730,  742, 1171, 1671]),
  array([0.6384, 0.8034, 0.6384, 0.6384]),
  742),
 (3,
  '(r1(s1,f1,25%,0.8))&(r2(s2,f1,25%,0.7))',
  {'r1': 3, 'r2': 3, 's1': 9075, 's2': 1290},
  array([1290, 4861, 3867, 6393]),
  array([1.064 , 1.1923, 1.064 , 1.064 ]),
  4861),
 (4,
  '(r1(s1,f1,75%,0.8))&(r2(s2,f1,25%,1.0))',
  {'r1': 0, 'r2': 0, 's1': 1258, 's2': 965},
  array([ 649, 4043, 1944, 2483]),
  array([1.1876, 1.2767, 1.3571, 1.3589]),
  2483)]

In [15]:
def parser_type0002(formula):
    return parser_type0001(formula)


format_type0002_str = """
[1. The statement of variables]
We have the following variables
1. f1

[2. The statement of conditions]
1.  {s1} {r1} f1; the necessity value is {alpha1}; the importance value is {beta1}. 
2.  {s2} {r2} f1; the necessity value is {alpha2}; the importance value is {beta2}. 


[3. The statement of assignments]
The possible value of f1 could be 
1 {choice1}
2 {choice2}
3 {choice3}
4 {choice4}

[Expected output]
Your output should be only the candidate index. DO NOT ADD YOUR EXPLANATION.
For example, if the answer is "2 test", your output MUST be the candidate index only. It MUST look like:
"2
<new line>
<new line>"

"""

outputs = []
for row in dataset_type0002:
    # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
    idx = row[0]
    alpha1, beta1, alpha2, beta2 = parser_type0002(row[1])
    r1 = entities[row[2]["r1"]]
    r2 = entities[row[2]["r2"]]
    s1 = entities[row[2]["s1"]]
    s2 = entities[row[2]["s2"]]
    
    choice1, choice2, choice3, choice4 = entities[row[3][0]],entities[row[3][1]], entities[row[3][2]], entities[row[3][3]]
    
    expected_answer = entities[row[5]]
    
    # Send Request to ChatGPT
    input_prompt = background_str + format_type0002_str.format(r1=r1,s1=s1,alpha1=alpha1,beta1=beta1,
                                                               r2=r2,s2=s2,alpha2=alpha2,beta2=beta2,
                                                               choice1=choice1, choice2=choice2, choice3=choice3, choice4=choice4)
    
#     print(input_prompt)
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
                    {"role": "user", "content": input_prompt}
                ]
    )


    reply = response['choices'][0]['message']['content']
    
    answer = None
    if  "1" in reply:
        answer = row[3][0]
    elif "2" in reply:
        answer = row[3][1]
    elif "3" in reply:
        answer = row[3][2]
    elif "4" in reply:
        answer = row[3][3]
    
    print("id:{}".format(idx), "reply: {}".format(reply), "mapped answer:", answer)
    outputs.append((idx, answer))

pickle_file = open("oct8/type0002_outputs.pickle", "wb")
pickle.dump(outputs, pickle_file)
pickle_file.close()

id:0 reply: 4 mapped answer: 8118
id:1 reply: 3 mapped answer: 516
id:2 reply: 3 mapped answer: 1171
id:3 reply: 2 mapped answer: 4861
id:4 reply: 4 mapped answer: 2483
id:5 reply: 2 mapped answer: 141
id:6 reply: 3 mapped answer: 6206
id:7 reply: 2 mapped answer: 3131
id:8 reply: 2 mapped answer: 1952
id:9 reply: 4 mapped answer: 1358
id:10 reply: 4 mapped answer: 1112
id:11 reply: 4 mapped answer: 1235
id:12 reply: 1 mapped answer: 722
id:13 reply: 1 mapped answer: 2787
id:14 reply: 1 mapped answer: 6041
id:15 reply: 3 mapped answer: 558
id:16 reply: 2 mapped answer: 482
id:17 reply: 4 mapped answer: 1247
id:18 reply: 1 mapped answer: 1660
id:19 reply: 2 mapped answer: 738
id:20 reply: 2 mapped answer: 3452
id:21 reply: 3 mapped answer: 573
id:22 reply: 3 mapped answer: 221


In [16]:
outputs

[(0, 8118),
 (1, 516),
 (2, 1171),
 (3, 4861),
 (4, 2483),
 (5, 141),
 (6, 6206),
 (7, 3131),
 (8, 1952),
 (9, 1358),
 (10, 1112),
 (11, 1235),
 (12, 722),
 (13, 2787),
 (14, 6041),
 (15, 558),
 (16, 482),
 (17, 1247),
 (18, 1660),
 (19, 738),
 (20, 3452),
 (21, 573),
 (22, 221)]

# type0004

In [17]:
dataset_type0004 = pickle.load(open(pickle_files["type0004"][0], "rb"))
print(len(dataset_type0004))
dataset_type0004[:5]

357


[(0,
  '(r1(s1,f1,25%,0.6))&(r2(e1,f1,25%,0.5))',
  {'r1': 0, 'r2': 0, 's1': 3996},
  array([2717, 9557,  240, 6255]),
  array([0.8944, 0.9057, 0.9256, 0.965 ]),
  6255),
 (1,
  '(r1(s1,f1,25%,0.8))&(r2(e1,f1,25%,0.2))',
  {'r1': 6, 'r2': 0, 's1': 844},
  array([11199,   136,  8397,  3305]),
  array([0.4022, 0.534 , 0.6649, 0.7102]),
  3305),
 (2,
  '(r1(s1,f1,75%,0.8))&(r2(e1,f1,75%,0.6))',
  {'r1': 0, 'r2': 0, 's1': 7862},
  array([6804, 4440,  699, 7878]),
  array([1.0802, 1.0893, 1.1674, 1.2376]),
  7878),
 (3,
  '(r1(s1,f1,25%,1.0))&(r2(e1,f1,75%,0.3))',
  {'r1': 4, 'r2': 0, 's1': 386},
  array([6332, 1517, 6138, 4233]),
  array([0.951 , 1.1783, 1.2224, 1.2564]),
  4233),
 (4,
  '(r1(s1,f1,25%,0.3))&(r2(e1,f1,25%,0.7))',
  {'r1': 0, 'r2': 0, 's1': 629},
  array([1468,  595, 1953,  259]),
  array([0.93  , 0.9378, 0.995 , 1.    ]),
  259)]

In [18]:
def parser_type0004(formula):
    return parser_type0001(formula)


format_type0004_str = """
[1. The statement of variables]
We have the following variables
1. f1
2. e1

[2. The statement of conditions]
1.  {s1} {r1} f1; the necessity value is {alpha1}; the importance value is {beta1}. 
2.  e1 {r2} f1; the necessity value is {alpha2}; the importance value is {beta2}. 


[3. The statement of assignments]
The possible value of f1 could be 
1 {choice1}
2 {choice2}
3 {choice3}
4 {choice4}

[Expected output]
Your output should be only the candidate index. DO NOT ADD YOUR EXPLANATION.
For example, if the answer is "2 test", your output MUST be the candidate index only. It MUST look like:
"2
<new line>
<new line>"

"""

outputs = []
for row in dataset_type0004[:150]:
    # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
    idx = row[0]
    alpha1, beta1, alpha2, beta2 = parser_type0004(row[1])
    r1 = entities[row[2]["r1"]]
    r2 = entities[row[2]["r2"]]
    s1 = entities[row[2]["s1"]]
    
    choice1, choice2, choice3, choice4 = entities[row[3][0]],entities[row[3][1]], entities[row[3][2]], entities[row[3][3]]
    
    expected_answer = entities[row[5]]
    
    # Send Request to ChatGPT
    input_prompt = background_str + format_type0004_str.format(r1=r1,s1=s1,alpha1=alpha1,beta1=beta1,
                                                               r2=r2,      alpha2=alpha2,beta2=beta2,
                                                               choice1=choice1, choice2=choice2, choice3=choice3, choice4=choice4)
    
#     print(input_prompt)
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
                    {"role": "user", "content": input_prompt}
                ]
    )


    reply = response['choices'][0]['message']['content']
    
    answer = None
    if  "1" in reply:
        answer = row[3][0]
    elif "2" in reply:
        answer = row[3][1]
    elif "3" in reply:
        answer = row[3][2]
    elif "4" in reply:
        answer = row[3][3]
    
    print("id:{}".format(idx), "reply: {}".format(reply), "mapped answer:", answer)
    outputs.append((idx, answer))


pickle_file = open("oct8/type0004_outputs.pickle", "wb")
pickle.dump(outputs, pickle_file)
pickle_file.close()

id:0 reply: 3 mapped answer: 240
id:1 reply: 1 mapped answer: 11199
id:2 reply: 4 mapped answer: 7878
id:3 reply: 1 mapped answer: 6332
id:4 reply: 1 mapped answer: 1468
id:5 reply: 4 mapped answer: 3805
id:6 reply: 2 mapped answer: 1288
id:7 reply: 2 mapped answer: 5363
id:8 reply: 1 mapped answer: 7060
id:9 reply: 2 mapped answer: 284
id:10 reply: 1 mapped answer: 2937
id:11 reply: 1 mapped answer: 113
id:12 reply: 1 mapped answer: 1160
id:13 reply: 2 mapped answer: 4354
id:14 reply: 1 mapped answer: 10535
id:15 reply: 3 mapped answer: 1129
id:16 reply: 2 mapped answer: 3372
id:17 reply: 4 mapped answer: 2550
id:18 reply: 3 mapped answer: 832
id:19 reply: 2 mapped answer: 12816
id:20 reply: 1 mapped answer: 869
id:21 reply: 2 mapped answer: 900
id:22 reply: 2 mapped answer: 226
id:23 reply: 3 mapped answer: 1447
id:24 reply: 3 mapped answer: 1750
id:25 reply: 2 mapped answer: 9918
id:26 reply: 3 mapped answer: 436
id:27 reply: 3 mapped answer: 5052
id:28 reply: 2 mapped answer: 5898


In [19]:
outputs

[(0, 240),
 (1, 11199),
 (2, 7878),
 (3, 6332),
 (4, 1468),
 (5, 3805),
 (6, 1288),
 (7, 5363),
 (8, 7060),
 (9, 284),
 (10, 2937),
 (11, 113),
 (12, 1160),
 (13, 4354),
 (14, 10535),
 (15, 1129),
 (16, 3372),
 (17, 2550),
 (18, 832),
 (19, 12816),
 (20, 869),
 (21, 900),
 (22, 226),
 (23, 1447),
 (24, 1750),
 (25, 9918),
 (26, 436),
 (27, 5052),
 (28, 5898),
 (29, 2872),
 (30, 13680),
 (31, 4980),
 (32, 4565),
 (33, 2510),
 (34, 4156),
 (35, 6540),
 (36, 3598),
 (37, 199),
 (38, 107),
 (39, 277),
 (40, 1144),
 (41, 3275),
 (42, 2483),
 (43, 11110),
 (44, 2388),
 (45, 1463),
 (46, 404),
 (47, 1924),
 (48, 4581),
 (49, 91),
 (50, 4529),
 (51, 7080),
 (52, 100),
 (53, 3625),
 (54, 1266),
 (55, 2366),
 (56, 2137),
 (57, 38),
 (58, 12112),
 (59, 1258),
 (60, 3548),
 (61, 5325),
 (62, 196),
 (63, 1793),
 (64, 422),
 (65, 9098),
 (66, 7384),
 (67, 2476),
 (68, 2290),
 (69, 8266),
 (70, 730),
 (71, 444),
 (72, 975),
 (73, 12816),
 (74, 1247),
 (75, 12914),
 (76, 2009),
 (77, 235),
 (78, 4316)

# type0005

In [20]:
dataset_type0005 = pickle.load(open(pickle_files["type0005"][0], "rb"))
print(len(dataset_type0005))
dataset_type0005[:5]

248


[(0,
  '(r1(s1,e1,25%,0.7))&((r2(e1,f1,25%,0.6))&(r3(e1,f1,25%,0.8)))',
  {'r1': 19, 'r2': 3, 'r3': 0, 's1': 6139},
  array([1927,  506, 2830, 8835]),
  array([1.5446, 1.6868, 1.693 , 1.8397]),
  8835),
 (1,
  '(r1(s1,e1,75%,0.7))&((r2(e1,f1,25%,0.8))&(r3(e1,f1,25%,0.6)))',
  {'r1': 0, 'r2': 11, 'r3': 0, 's1': 596},
  array([1956,   69, 3959, 2022]),
  array([1.4004, 1.4207, 1.438 , 1.5077]),
  2022),
 (2,
  '(r1(s1,e1,25%,0.9))&((r2(e1,f1,75%,0.2))&(r3(e1,f1,25%,0.1)))',
  {'r1': 0, 'r2': 0, 'r3': 3, 's1': 595},
  array([3455, 3677, 1054, 4473]),
  array([1.0346, 1.0394, 1.0529, 1.0712]),
  4473),
 (3,
  '(r1(s1,e1,75%,0.8))&((r2(e1,f1,75%,0.5))&(r3(e1,f1,75%,0.8)))',
  {'r1': 0, 'r2': 9, 'r3': 0, 's1': 2661},
  array([5794, 7551, 4672, 1024]),
  array([1.5812, 1.6975, 1.8747, 1.9605]),
  1024),
 (4,
  '(r1(s1,e1,25%,0.4))&((r2(e1,f1,75%,0.7))&(r3(e1,f1,25%,0.4)))',
  {'r1': 0, 'r2': 2, 'r3': 0, 's1': 7887},
  array([ 278, 5908,   21, 2009]),
  array([1.2505, 1.2898, 1.3008, 1.3086]),

In [21]:
def parser_type0005(formula):
    f = formula.split("&")
    first_formula = f[0][1:-1]
    second_formula = f[1][2:-1]
    third_formula = f[2][1:-2]
    r1, s1, e1, alpha1, beta1 = parser_single(first_formula)
    r2, e1, f1, alpha2, beta2 = parser_single(second_formula)
    r3, e1, f1, alpha3, beta3 = parser_single(third_formula)
    
    return alpha1, beta1, alpha2, beta2, alpha3, beta3

In [22]:
format_type0005_str = """
[1. The statement of variables]
We have the following variables
1. f1
2. e1

[2. The statement of conditions]
1.  {s1} {r1} e1; the necessity value is {alpha1}; the importance value is {beta1}. 
2.  e1 {r2} f1; the necessity value is {alpha2}; the importance value is {beta2}. 
3.  e1 {r3} f1; the necessity value is {alpha3}; the importance value is {beta3}. 


[3. The statement of assignments]
The possible value of f1 could be 
1 {choice1}
2 {choice2}
3 {choice3}
4 {choice4}

[Expected output]
Your output should be ONLY the candidate index. DO NOT ADD YOUR EXPLANATION.
For example, if the answer is "2 test", your output MUST be the candidate index ONLY. It MUST look like:
"2
<new line>
<new line>"

"""

outputs = []
for row in dataset_type0005[:150]:
    # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
    idx = row[0]
    alpha1, beta1, alpha2, beta2, alpha3, beta3 = parser_type0005(row[1])
    r1 = entities[row[2]["r1"]]
    r2 = entities[row[2]["r2"]]
    r3 = entities[row[2]["r3"]]
    
    s1 = entities[row[2]["s1"]]
    
    choice1, choice2, choice3, choice4 = entities[row[3][0]],entities[row[3][1]], entities[row[3][2]], entities[row[3][3]]
    
    expected_answer = entities[row[5]]
    
    # Send Request to ChatGPT
    input_prompt = background_str + format_type0005_str.format(r1=r1,s1=s1,alpha1=alpha1,beta1=beta1,
                                                               r2=r2,      alpha2=alpha2,beta2=beta2,
                                                               r3=r3,      alpha3=alpha3,beta3=beta3,
                                                               choice1=choice1, choice2=choice2, choice3=choice3, choice4=choice4)
    
#     print(input_prompt)
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
                    {"role": "user", "content": input_prompt}
                ]
    )


    reply = response['choices'][0]['message']['content']
    
    answer = None
    if  "1" in reply:
        answer = row[3][0]
    elif "2" in reply:
        answer = row[3][1]
    elif "3" in reply:
        answer = row[3][2]
    elif "4" in reply:
        answer = row[3][3]
    
    print("id:{}".format(idx), "reply: {}".format(reply), "mapped answer:", answer)
    outputs.append((idx, answer))

print(outputs)
pickle_file = open("oct8/type0005_outputs.pickle", "wb")
pickle.dump(outputs, pickle_file)
pickle_file.close()

id:0 reply: 1 mapped answer: 1927
id:1 reply: 1 mapped answer: 1956
id:2 reply: 2 mapped answer: 3677
id:3 reply: 4 mapped answer: 1024
id:4 reply: 1 mapped answer: 278
id:5 reply: 3 mapped answer: 3253
id:6 reply: 4 mapped answer: 731
id:7 reply: 3 mapped answer: 7220
id:8 reply: 2 mapped answer: 976
id:9 reply: 3 mapped answer: 829
id:10 reply: 3 mapped answer: 6978
id:11 reply: 2 mapped answer: 4738
id:12 reply: 2 mapped answer: 7328
id:13 reply: 1 mapped answer: 3643
id:14 reply: 2 mapped answer: 4392
id:15 reply: 3 mapped answer: 1559
id:16 reply: 1 mapped answer: 7602
id:17 reply: 1 mapped answer: 4412
id:18 reply: 1 mapped answer: 1519
id:19 reply: 3 mapped answer: 185
id:20 reply: 1 mapped answer: 3088
id:21 reply: 3 mapped answer: 2874
id:22 reply: 3 mapped answer: 1385
id:23 reply: 2 mapped answer: 2599
id:24 reply: 3 mapped answer: 1921
id:25 reply: 1 mapped answer: 422
id:26 reply: 1 mapped answer: 1801
id:27 reply: 3 mapped answer: 3875
id:28 reply: 3 mapped answer: 671
id

In [24]:
outputs

[(0, 1927),
 (1, 1956),
 (2, 3677),
 (3, 1024),
 (4, 278),
 (5, 3253),
 (6, 731),
 (7, 7220),
 (8, 976),
 (9, 829),
 (10, 6978),
 (11, 4738),
 (12, 7328),
 (13, 3643),
 (14, 4392),
 (15, 1559),
 (16, 7602),
 (17, 4412),
 (18, 1519),
 (19, 185),
 (20, 3088),
 (21, 2874),
 (22, 1385),
 (23, 2599),
 (24, 1921),
 (25, 422),
 (26, 1801),
 (27, 3875),
 (28, 671),
 (29, 2592),
 (30, 2033),
 (31, 11158),
 (32, 776),
 (33, 301),
 (34, 966),
 (35, 1936),
 (36, 3179),
 (37, 13978),
 (38, 3436),
 (39, 2067),
 (40, 2483),
 (41, 712),
 (42, 655),
 (43, 7271),
 (44, 768),
 (45, 4406),
 (46, 2798),
 (47, 3710),
 (48, 4006),
 (49, 4174),
 (50, 1703),
 (51, 8118),
 (52, 4722),
 (53, 9940),
 (54, 1463),
 (55, 260),
 (56, 9441),
 (57, 185),
 (58, 2343),
 (59, 1128),
 (60, 2522),
 (61, 3570),
 (62, 1112),
 (63, 3875),
 (64, 1601),
 (65, 1750),
 (66, 3225),
 (67, 386),
 (68, 8449),
 (69, 3483),
 (70, 3947),
 (71, 4174),
 (72, 5564),
 (73, 1734),
 (74, 263),
 (75, 3225),
 (76, 6938),
 (77, 550),
 (78, 6041),

# type0008

In [25]:
dataset_type0008 = pickle.load(open(pickle_files["type0008"][0], "rb"))
print(len(dataset_type0008))
dataset_type0008[:5]

316


[(0,
  '(r1(s1,e1,75%,0.5))&((r2(s2,e1,75%,0.4))&(r3(e1,f1,25%,0.7)))',
  {'r1': 0, 'r2': 0, 'r3': 3, 's1': 3822, 's2': 2002},
  array([4879,  132,  913, 2557]),
  array([1.0065, 1.1349, 1.2633, 1.1349]),
  913),
 (1,
  '(r1(s1,e1,25%,0.3))&((r2(s2,e1,75%,0.9))&(r3(e1,f1,25%,0.4)))',
  {'r1': 0, 'r2': 9, 'r3': 0, 's1': 6404, 's2': 1708},
  array([ 773, 1569, 1466, 6546]),
  array([1.2844, 1.2979, 1.3033, 1.3507]),
  6546),
 (2,
  '(r1(s1,e1,75%,0.2))&((r2(s2,e1,25%,0.5))&(r3(e1,f1,25%,0.6)))',
  {'r1': 0, 'r2': 0, 'r3': 0, 's1': 8131, 's2': 8131},
  array([5004, 2250, 3370, 2873]),
  array([1.0831, 1.0933, 1.0939, 1.0965]),
  2873),
 (3,
  '(r1(s1,e1,75%,0.4))&((r2(s2,e1,25%,0.9))&(r3(e1,f1,75%,0.2)))',
  {'r1': 0, 'r2': 0, 'r3': 0, 's1': 4196, 's2': 4196},
  array([13095,   101,  2366,   161]),
  array([1.117 , 1.1193, 1.1198, 1.1221]),
  161),
 (4,
  '(r1(s1,e1,25%,0.6))&((r2(s2,e1,75%,0.7))&(r3(e1,f1,75%,0.5)))',
  {'r1': 0, 'r2': 0, 'r3': 0, 's1': 5209, 's2': 5209},
  array([13645,

In [26]:
def parser_type0008(formula):
    return parser_type0005(formula)

format_type0008_str = """
[1. The statement of variables]
We have the following variables
1. f1
2. e1

[2. The statement of conditions]
1.  {s1} {r1} e1; the necessity value is {alpha1}; the importance value is {beta1}. 
2.  {s2} {r2} e1; the necessity value is {alpha2}; the importance value is {beta2}. 
3.  e1 {r3} f1; the necessity value is {alpha3}; the importance value is {beta3}. 


[3. The statement of assignments]
The possible value of f1 could be 
1 {choice1}
2 {choice2}
3 {choice3}
4 {choice4}

[Expected output]
Your output should be ONLY the candidate index. DO NOT ADD YOUR EXPLANATION.
For example, if the answer is "2 test", your output MUST be the candidate index ONLY. It MUST look like:
"2
<new line>
<new line>"

"""


outputs = []
for row in dataset_type0008[:150]:
    # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
    idx = row[0]
    alpha1, beta1, alpha2, beta2, alpha3, beta3 = parser_type0008(row[1])
    r1 = entities[row[2]["r1"]]
    r2 = entities[row[2]["r2"]]
    r3 = entities[row[2]["r3"]]
    
    s1 = entities[row[2]["s1"]]
    s2 = entities[row[2]["s2"]]
    
    choice1, choice2, choice3, choice4 = entities[row[3][0]],entities[row[3][1]], entities[row[3][2]], entities[row[3][3]]
    
    expected_answer = entities[row[5]]
    
    # Send Request to ChatGPT
    input_prompt = background_str + format_type0008_str.format(r1=r1,s1=s1,alpha1=alpha1,beta1=beta1,
                                                               r2=r2,s2=s2,alpha2=alpha2,beta2=beta2,
                                                               r3=r3,      alpha3=alpha3,beta3=beta3,
                                                               choice1=choice1, choice2=choice2, choice3=choice3, choice4=choice4)
    
#     print(input_prompt)
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
                    {"role": "user", "content": input_prompt}
                ]
    )


    reply = response['choices'][0]['message']['content']
    
    answer = None
    if  "1" in reply:
        answer = row[3][0]
    elif "2" in reply:
        answer = row[3][1]
    elif "3" in reply:
        answer = row[3][2]
    elif "4" in reply:
        answer = row[3][3]
    
    print("id:{}".format(idx), "reply: {}".format(reply), "mapped answer:", answer)
    outputs.append((idx, answer))
    
pickle_file = open("oct8/type0008_outputs.pickle", "wb")
pickle.dump(outputs, pickle_file)
pickle_file.close()

id:0 reply: 3 mapped answer: 913
id:1 reply: 2 mapped answer: 1569
id:2 reply: 2 mapped answer: 2250
id:3 reply: 2 mapped answer: 101
id:4 reply: 2 mapped answer: 1025
id:5 reply: 1 mapped answer: 1067
id:6 reply: 3 mapped answer: 969
id:7 reply: 1 mapped answer: 62
id:8 reply: 1 mapped answer: 7060
id:9 reply: 4 mapped answer: 4684
id:10 reply: 2 mapped answer: 7561
id:11 reply: 3 mapped answer: 1122
id:12 reply: 3 mapped answer: 8449
id:13 reply: 2 mapped answer: 3626
id:14 reply: 2 mapped answer: 12638
id:15 reply: 3 mapped answer: 2002
id:16 reply: 1 mapped answer: 1316
id:17 reply: 1 mapped answer: 1912
id:18 reply: 1 mapped answer: 13657
id:19 reply: 3 mapped answer: 10171
id:20 reply: 1 mapped answer: 2483
id:21 reply: 2 mapped answer: 38
id:22 reply: 1 mapped answer: 730
id:23 reply: 2 mapped answer: 976
id:24 reply: 2 mapped answer: 259
id:25 reply: 3 mapped answer: 7261
id:26 reply: 1 mapped answer: 3806
id:27 reply: 4 mapped answer: 1831
id:28 reply: 3 mapped answer: 3893
id

In [27]:
outputs

[(0, 913),
 (1, 1569),
 (2, 2250),
 (3, 101),
 (4, 1025),
 (5, 1067),
 (6, 969),
 (7, 62),
 (8, 7060),
 (9, 4684),
 (10, 7561),
 (11, 1122),
 (12, 8449),
 (13, 3626),
 (14, 12638),
 (15, 2002),
 (16, 1316),
 (17, 1912),
 (18, 13657),
 (19, 10171),
 (20, 2483),
 (21, 38),
 (22, 730),
 (23, 976),
 (24, 259),
 (25, 7261),
 (26, 3806),
 (27, 1831),
 (28, 3893),
 (29, 1926),
 (30, 6755),
 (31, 1312),
 (32, 5334),
 (33, 82),
 (34, 5370),
 (35, 8012),
 (36, 1309),
 (37, 340),
 (38, 11402),
 (39, 2450),
 (40, 11435),
 (41, 91),
 (42, 9431),
 (43, 7069),
 (44, 2717),
 (45, 13707),
 (46, 3014),
 (47, 759),
 (48, 968),
 (49, 2483),
 (50, 1893),
 (51, 4081),
 (52, 12008),
 (53, 1063),
 (54, 5757),
 (55, 22),
 (56, 2717),
 (57, 5810),
 (58, 2142),
 (59, 1522),
 (60, 2653),
 (61, 1630),
 (62, 6102),
 (63, 939),
 (64, 5898),
 (65, 2483),
 (66, 992),
 (67, 686),
 (68, 4529),
 (69, 3151),
 (70, 3572),
 (71, 738),
 (72, 11609),
 (73, 1671),
 (74, 865),
 (75, 8261),
 (76, 112),
 (77, 418),
 (78, 13623),


# type0009

In [28]:
dataset_type0009 = pickle.load(open(pickle_files["type0009"][0], "rb"))
print(len(dataset_type0009))
dataset_type0009[:5]

96


[(0,
  '(r1(s1,e1,75%,0.5))&((r2(s2,e1,75%,0.6))&((r3(e1,f1,75%,0.8))&(r4(e1,f1,75%,0.4))))',
  {'r1': 2, 'r2': 0, 'r3': 0, 'r4': 3, 's1': 13442, 's2': 10451},
  array([1154, 5310, 3225, 2208]),
  array([1.7964, 1.8331, 1.9127, 1.7964]),
  3225),
 (1,
  '(r1(s1,e1,75%,0.8))&((r2(s2,e1,25%,0.6))&((r3(e1,f1,25%,0.5))&(r4(e1,f1,25%,0.6))))',
  {'r1': 0, 'r2': 0, 'r3': 4, 'r4': 0, 's1': 10306, 's2': 10306},
  array([2487, 1908,  741,   51]),
  array([1.7733, 1.8937, 1.998 , 2.0843]),
  51),
 (2,
  '(r1(s1,e1,25%,0.7))&((r2(s2,e1,25%,0.6))&((r3(e1,f1,25%,0.1))&(r4(e1,f1,75%,0.8))))',
  {'r1': 0, 'r2': 0, 'r3': 3, 'r4': 0, 's1': 1799, 's2': 4272},
  array([1741, 4670, 6652, 8678]),
  array([1.5788, 1.7946, 1.7974, 1.5788]),
  6652),
 (3,
  '(r1(s1,e1,75%,0.2))&((r2(s2,e1,25%,0.1))&((r3(e1,f1,25%,0.7))&(r4(e1,f1,25%,0.5))))',
  {'r1': 4, 'r2': 4, 'r3': 2, 'r4': 0, 's1': 1423, 's2': 1423},
  array([  537, 12710,   187,  1882]),
  array([1.2192, 1.2214, 1.2308, 1.2474]),
  1882),
 (4,
  '(r1(s1

In [29]:
def parser_type0009(formula):
    f = formula.split("&")
    first_formula = f[0][1:-1]
    second_formula = f[1][2:-1]
    third_formula = f[2][2:-1]
    forth_formula = f[3][1:-3]
    r1, s1, e1, alpha1, beta1 = parser_single(first_formula)
    r2, s2, e1, alpha2, beta2 = parser_single(second_formula)
    r3, e1, f1, alpha3, beta3 = parser_single(third_formula)
    r4, e1, f1, alpha4, beta4 = parser_single(forth_formula)
    return alpha1, beta1, alpha2, beta2, alpha3, beta3, alpha4, beta4

In [30]:
format_type0009_str = """
[1. The statement of variables]
We have the following variables
1. f1
2. e1

[2. The statement of conditions]
1.  {s1} {r1} e1; the necessity value is {alpha1}; the importance value is {beta1}. 
2.  {s2} {r2} e1; the necessity value is {alpha2}; the importance value is {beta2}. 
3.  e1 {r3} f1; the necessity value is {alpha3}; the importance value is {beta3}. 
4.  e1 {r4} f1; the necessity value is {alpha4}; the importance value is {beta4}. 


[3. The statement of assignments]
The possible value of f1 could be 
1 {choice1}
2 {choice2}
3 {choice3}
4 {choice4}

[Expected output]
Your output should be ONLY the candidate index. DO NOT ADD YOUR EXPLANATION.
For example, if the answer is "2 test", your output MUST be the candidate index ONLY. It MUST look like:
"2
<new line>
<new line>"

"""

outputs = []
for row in dataset_type0009:
    # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
    idx = row[0]
    alpha1, beta1, alpha2, beta2, alpha3, beta3, alpha4, beta4 = parser_type0009(row[1])
    r1 = entities[row[2]["r1"]]
    r2 = entities[row[2]["r2"]]
    r3 = entities[row[2]["r3"]]
    r4 = entities[row[2]["r4"]]
    
    s1 = entities[row[2]["s1"]]
    s2 = entities[row[2]["s2"]]
    
    choice1, choice2, choice3, choice4 = entities[row[3][0]],entities[row[3][1]], entities[row[3][2]], entities[row[3][3]]
    
    expected_answer = entities[row[5]]
    
    # Send Request to ChatGPT
    input_prompt = background_str + format_type0009_str.format(r1=r1,s1=s1,alpha1=alpha1,beta1=beta1,
                                                               r2=r2,s2=s2,alpha2=alpha2,beta2=beta2,
                                                               r3=r3,      alpha3=alpha3,beta3=beta3,
                                                               r4=r4,      alpha4=alpha4,beta4=beta4,
                                                               choice1=choice1, choice2=choice2, choice3=choice3, choice4=choice4)
    
#     print(input_prompt)
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
                    {"role": "user", "content": input_prompt}
                ]
    )


    reply = response['choices'][0]['message']['content']
    
    answer = None
    if  "1" in reply:
        answer = row[3][0]
    elif "2" in reply:
        answer = row[3][1]
    elif "3" in reply:
        answer = row[3][2]
    elif "4" in reply:
        answer = row[3][3]
    
    print("id:{}".format(idx), "reply: {}".format(reply), "mapped answer:", answer)
    outputs.append((idx, answer))
    
pickle_file = open("oct8/type0009_outputs.pickle", "wb")
pickle.dump(outputs, pickle_file)
pickle_file.close()

id:0 reply: 1 mapped answer: 1154
id:1 reply: 2 mapped answer: 1908
id:2 reply: 3 mapped answer: 6652
id:3 reply: 1 mapped answer: 537
id:4 reply: 2 mapped answer: 5994
id:5 reply: 3 mapped answer: 11720
id:6 reply: 3 mapped answer: 1398
id:7 reply: 4 mapped answer: 8118
id:8 reply: 1 mapped answer: 1691
id:9 reply: 1 mapped answer: 4385
id:10 reply: 1 mapped answer: 3651
id:11 reply: 2 mapped answer: 6726
id:12 reply: 1 mapped answer: 6042
id:13 reply: 1 mapped answer: 1639
id:14 reply: 3 mapped answer: 8449
id:15 reply: 2 mapped answer: 4557
id:16 reply: 1 mapped answer: 4607
id:17 reply: 3 mapped answer: 9389
id:18 reply: 4 mapped answer: 3371
id:19 reply: 1 mapped answer: 75
id:20 reply: 2 mapped answer: 975
id:21 reply: 2 mapped answer: 2179
id:22 reply: 4 mapped answer: 1151
id:23 reply: 3 mapped answer: 1010
id:24 reply: 2 mapped answer: 416
id:25 reply: 2 mapped answer: 946
id:26 reply: 3 mapped answer: 6509
id:27 reply: 4 mapped answer: 422
id:28 reply: 3 mapped answer: 2145
i

In [31]:
outputs

[(0, 1154),
 (1, 1908),
 (2, 6652),
 (3, 537),
 (4, 5994),
 (5, 11720),
 (6, 1398),
 (7, 8118),
 (8, 1691),
 (9, 4385),
 (10, 3651),
 (11, 6726),
 (12, 6042),
 (13, 1639),
 (14, 8449),
 (15, 4557),
 (16, 4607),
 (17, 9389),
 (18, 3371),
 (19, 75),
 (20, 975),
 (21, 2179),
 (22, 1151),
 (23, 1010),
 (24, 416),
 (25, 946),
 (26, 6509),
 (27, 422),
 (28, 2145),
 (29, 4135),
 (30, 9559),
 (31, 448),
 (32, 4185),
 (33, 9973),
 (34, 223),
 (35, 649),
 (36, 1882),
 (37, 6439),
 (38, 1797),
 (39, 1841),
 (40, 8453),
 (41, 284),
 (42, 5324),
 (43, 4467),
 (44, 108),
 (45, 815),
 (46, 3241),
 (47, 4135),
 (48, 6829),
 (49, 4677),
 (50, 8247),
 (51, 785),
 (52, 5439),
 (53, 3894),
 (54, 4953),
 (55, 4085),
 (56, 8754),
 (57, 1967),
 (58, 74),
 (59, 5288),
 (60, 12),
 (61, 51),
 (62, 193),
 (63, 422),
 (64, 1128),
 (65, 4565),
 (66, 1756),
 (67, 1629),
 (68, 1540),
 (69, 8486),
 (70, 6938),
 (71, 2369),
 (72, 4185),
 (73, 2428),
 (74, 6652),
 (75, 4385),
 (76, 3965),
 (77, 9832),
 (78, 5800),
 (79